# Week 3: Calibration, Geometry, and EP Analysis

<a href="https://colab.research.google.com/github/elenaajayi/spec-gap-activation-probe/blob/main/notebooks/03_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loads Week 2 collusion probe artifacts and produces calibration diagnostics,
reliability diagrams, PCA geometry visualizations, exemplar partitioning summaries,
and a preliminary writeup.

**Inputs:**

- `week2_collusion_probe_results.json`
- `week2_collusion_probe_activations.npz`
- optional: `week2_ep_results.json`, `week2_collusion_probe_responses.json`

If running standalone, upload these files or re-run Week 2 first.


In [0]:
# Artifact directory setup and Week 2 artifact load
import os
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ARTIFACT_ROOT = Path("/content/drive/MyDrive/spec-gap-activation-probe/artifacts")
except Exception:
    DEFAULT_ARTIFACT_ROOT = Path.cwd() / "artifacts"

ARTIFACT_ROOT = Path(os.environ.get("SPEC_GAP_ARTIFACT_ROOT", DEFAULT_ARTIFACT_ROOT))
WEEK2_ARTIFACT_DIR = ARTIFACT_ROOT / "02_collusion_probe"
ARTIFACT_DIR = ARTIFACT_ROOT / "03_analysis"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"Reading Week 2 artifacts from: {WEEK2_ARTIFACT_DIR}")
print(f"This notebook writes to: {ARTIFACT_DIR}")

REQUIRED_ARTIFACTS = [
    "week2_collusion_probe_results.json",
    "week2_collusion_probe_activations.npz",
]
OPTIONAL_ARTIFACTS = [
    "week2_ep_results.json",
    "week2_collusion_probe_responses.json",
]

for fname in REQUIRED_ARTIFACTS + OPTIONAL_ARTIFACTS:
    src = WEEK2_ARTIFACT_DIR / fname
    dst = Path(fname)
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"Copied {fname} from {WEEK2_ARTIFACT_DIR}")
    elif dst.exists():
        print(f"{fname} already in working directory")
    elif fname in REQUIRED_ARTIFACTS:
        raise FileNotFoundError(f"Required artifact not found: {src}. Run notebook 02 first and save artifacts.")
    else:
        print(f"Optional artifact {fname} not found at {src}; continuing without it")


In [0]:
# 1. Load Week 2 results (JSON + NPZ sidecar)
import json
import numpy as np
import matplotlib.pyplot as plt
import os

with open("week2_collusion_probe_results.json") as f:
    results = json.load(f)

npz = np.load(results["activations_file"])
activations = {}
for key in npz.files:
    if key.startswith("layer_"):
        layer_num = key.split("_")[1]
        activations[layer_num] = npz[key]
labels = np.array(results["labels"])

ep_results = None
if os.path.exists("week2_ep_results.json"):
    with open("week2_ep_results.json") as f:
        ep_results = json.load(f)

responses = None
if os.path.exists("week2_collusion_probe_responses.json"):
    with open("week2_collusion_probe_responses.json") as f:
        responses = json.load(f)

print(f"Experiment: {results['experiment']}")
print(f"Model: {results['model']}")
print(f"Date: {results['date']}")
print(f"Prompts: {results['n_prompts']} ({results['n_colluder']} colluder, {results['n_honest']} honest)")
print(f"PCA components: {results['pca_components']}")
print(f"Activations loaded: {sorted(activations.keys(), key=int)}, shape: {next(iter(activations.values())).shape}")
print(f"Generation temp: {results.get('generation_temperature', 'not recorded')}; max_new_tokens: {results.get('max_new_tokens', 'not recorded')}")
print(f"EP loaded: {ep_results is not None}; responses loaded: {responses is not None}")


In [0]:
# 2. Summary table
layers = sorted(results["stratified_cv"].keys(), key=int)

print(f"{'Layer':<8} {'AUROC (5-fold)':<22} {'AUROC (LSO)':<22} {'Brier':<10} {'ECE':<10} {'cos(p,DIM)':<12}")
print("-" * 84)
for layer in layers:
    cv = results["stratified_cv"][layer]
    lso = results["leave_scenario_out_cv"][layer]
    d = results["directions"][layer]
    print(f"{layer:<8} {cv['auroc_mean']:.3f} +/- {cv['auroc_std']:.3f}        "
          f"{lso['auroc_mean']:.3f} +/- {lso['auroc_std']:.3f}        "
          f"{cv['brier_mean']:.3f}     {cv['ece_mean']:.3f}     {d['cosine_sim']:.3f}")

In [0]:
# 2b. Re-run stratified CV to collect per-sample predictions
# (the save format doesn't store these, so we re-fit with the same settings)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as PCA_CV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

cv_predictions = {}
pca_components = results["pca_components"]

for layer in layers:
    X = activations[layer]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_true_all, y_prob_all = [], []
    for train_idx, test_idx in skf.split(X, labels):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA_CV(n_components=pca_components)),
            ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
        ])
        pipe.fit(X[train_idx], labels[train_idx])
        y_prob = pipe.predict_proba(X[test_idx])[:, 1]
        y_true_all.extend(labels[test_idx].tolist())
        y_prob_all.extend(y_prob.tolist())
    cv_predictions[layer] = {
        "y_true": np.array(y_true_all),
        "y_prob": np.array(y_prob_all),
    }
    print(f"Layer {layer}: {len(y_true_all)} predictions collected")

In [0]:
# 2c. Bootstrap 95% CIs for 5-fold AUROC (1000 resamples of pooled test predictions)
from sklearn.metrics import roc_auc_score

COMMITTED_LAYERS = ["16", "20", "24"]

bootstrap_cis = {}
n_bootstrap = 1000
rng = np.random.RandomState(42)

for layer in layers:
    preds = cv_predictions[layer]
    y_true = preds["y_true"]
    y_prob = preds["y_prob"]
    n = len(y_true)

    boot_aurocs = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_true[idx], y_prob[idx]))

    boot_aurocs = np.array(boot_aurocs)
    bootstrap_cis[layer] = {
        "mean": float(np.mean(boot_aurocs)),
        "ci_lo": float(np.percentile(boot_aurocs, 2.5)),
        "ci_hi": float(np.percentile(boot_aurocs, 97.5)),
    }

print(f"{'Layer':<8} {'AUROC':<8} {'Bootstrap 95% CI':<24} {'Width':<8}")
print("-" * 48)
for layer in layers:
    ci = bootstrap_cis[layer]
    w = ci["ci_hi"] - ci["ci_lo"]
    marker = " *" if layer in COMMITTED_LAYERS else ""
    print(f"{layer:<8} {ci['mean']:.3f}   [{ci['ci_lo']:.3f}, {ci['ci_hi']:.3f}]       {w:.3f}{marker}")
print("\n* = committed analysis layer")

In [0]:
# 3. Reliability diagrams (calibration)
fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 4.5))
if len(layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    cv = results["stratified_cv"][layer]
    y_true = cv_predictions[layer]["y_true"]
    y_prob = cv_predictions[layer]["y_prob"]

    n_bins = 10
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob <= hi) if i == n_bins - 1 else (y_prob >= lo) & (y_prob < hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0)
            bin_confs.append((lo + hi) / 2)
            bin_counts.append(0)
        else:
            bin_accs.append(float(y_true[mask].mean()))
            bin_confs.append(float(y_prob[mask].mean()))
            bin_counts.append(int(count))

    bin_centers = [(bin_edges[i] + bin_edges[i + 1]) / 2 for i in range(n_bins)]
    width = 1.0 / n_bins

    ax.bar(bin_centers, bin_accs, width=width * 0.9, alpha=0.7, color="#2196F3", label="Accuracy")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.set_title(f"Layer {layer} (Brier={cv['brier_mean']:.3f}, ECE={cv['ece_mean']:.3f})")

plt.suptitle("Reliability Diagrams — Stratified 5-Fold CV", y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_reliability_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 3b. Reliability diagrams for committed layers (16, 20, 24) with miscalibration gap
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, layer in zip(axes, COMMITTED_LAYERS):
    y_true = cv_predictions[layer]["y_true"]
    y_prob = cv_predictions[layer]["y_prob"]
    cv = results["stratified_cv"][layer]

    n_bins = 10
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob <= hi) if i == n_bins - 1 else (y_prob >= lo) & (y_prob < hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(np.nan)
            bin_confs.append((lo + hi) / 2)
            bin_counts.append(0)
        else:
            bin_accs.append(float(y_true[mask].mean()))
            bin_confs.append(float(y_prob[mask].mean()))
            bin_counts.append(int(count))

    bin_centers = [(bin_edges[i] + bin_edges[i + 1]) / 2 for i in range(n_bins)]
    w = 1.0 / n_bins

    ax.bar(bin_centers, bin_accs, width=w * 0.85, alpha=0.7, color="#2196F3",
           label="Observed frequency")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.7, label="Perfect calibration")

    for j in range(n_bins):
        if bin_counts[j] > 0 and not np.isnan(bin_accs[j]):
            gap_lo = min(bin_accs[j], bin_centers[j])
            gap_hi = max(bin_accs[j], bin_centers[j])
            ax.fill_between(
                [bin_centers[j] - w * 0.425, bin_centers[j] + w * 0.425],
                gap_lo, gap_hi, alpha=0.25, color="red",
            )

    ax2 = ax.twinx()
    ax2.bar(bin_centers, bin_counts, width=w * 0.85, alpha=0.12, color="gray")
    ax2.set_ylabel("Count", fontsize=9, color="gray")
    ax2.tick_params(axis="y", labelcolor="gray", labelsize=8)

    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"Layer {layer}\nBrier={cv['brier_mean']:.3f}, ECE={cv['ece_mean']:.3f}")
    ax.legend(fontsize=8, loc="upper left")

plt.suptitle("Reliability Diagrams — Committed Layers (5-Fold CV)", y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_reliability_committed.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 4. PCA geometry — scatter plots colored by label
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 4.5))
if len(layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    X = activations[layer]

    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)
    ev = pca.explained_variance_ratio_

    ax.scatter(X_2d[labels == 0, 0], X_2d[labels == 0, 1], alpha=0.6, s=30, label="Honest", c="#4CAF50")
    ax.scatter(X_2d[labels == 1, 0], X_2d[labels == 1, 1], alpha=0.6, s=30, label="Colluder", c="#F44336")
    ax.set_xlabel(f"PC1 ({ev[0]:.1%})")
    ax.set_ylabel(f"PC2 ({ev[1]:.1%})")
    ax.set_title(f"Layer {layer}")
    ax.legend(fontsize=9)

plt.suptitle("PCA of Residual Stream Activations by Role", y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_pca_geometry.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 5. Probe direction vs top PCA components
print("Alignment of probe direction with top PCA components:\n")
for layer in layers:
    d = results["directions"][layer]
    probe_dir = np.array(d["probe_direction"])
    dim_dir = np.array(d["dim_direction"])

    X = activations[layer]
    pca = PCA(n_components=5)
    pca.fit(X)

    print(f"Layer {layer}:")
    print(f"  cos(probe, DIM) = {d['cosine_sim']:.3f}")
    print(f"  Probe norm = {d['probe_norm']:.3f}, DIM norm = {d['dim_norm']:.3f}")
    for k in range(5):
        cos_probe_pc = float(np.dot(probe_dir, pca.components_[k]) /
                             (np.linalg.norm(probe_dir) * np.linalg.norm(pca.components_[k])))
        cos_dim_pc = float(np.dot(dim_dir, pca.components_[k]) /
                           (np.linalg.norm(dim_dir) * np.linalg.norm(pca.components_[k])))
        print(f"  cos(probe, PC{k+1}) = {cos_probe_pc:.3f}   cos(DIM, PC{k+1}) = {cos_dim_pc:.3f}   "
              f"var explained = {pca.explained_variance_ratio_[k]:.3f}")
    print()

In [0]:
# 5b. Explained variance curves and effective dimensionality
from sklearn.decomposition import PCA as PCA_EV

eff_dims = {}
for layer in layers:
    X = activations[layer]
    n_comp = min(50, X.shape[0] - 1)
    pca_ev = PCA_EV(n_components=n_comp)
    pca_ev.fit(X)
    cumvar = np.cumsum(pca_ev.explained_variance_ratio_)
    eff_dims[layer] = {
        "d90": int(np.searchsorted(cumvar, 0.90) + 1),
        "d95": int(np.searchsorted(cumvar, 0.95) + 1),
        "cumvar": cumvar,
    }

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

for layer in COMMITTED_LAYERS:
    cumvar = eff_dims[layer]["cumvar"]
    ax1.plot(range(1, len(cumvar) + 1), cumvar, label=f"Layer {layer}", linewidth=1.5)
ax1.axhline(0.95, color="gray", linestyle=":", alpha=0.5, label="95%")
ax1.axhline(0.90, color="gray", linestyle="--", alpha=0.3, label="90%")
ax1.set_xlabel("Number of components")
ax1.set_ylabel("Cumulative explained variance")
ax1.set_title("Explained Variance — Committed Layers")
ax1.legend(fontsize=9)
ax1.set_xlim(1, 50)

layer_ints = [int(l) for l in layers]
d90 = [eff_dims[l]["d90"] for l in layers]
d95 = [eff_dims[l]["d95"] for l in layers]
ax2.plot(layer_ints, d90, "o-", label="90% variance", color="#2196F3")
ax2.plot(layer_ints, d95, "s-", label="95% variance", color="#FF9800")
for l in [16, 20, 24]:
    ax2.axvspan(l - 0.3, l + 0.3, alpha=0.08, color="orange")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Number of components")
ax2.set_title("Effective Dimensionality by Layer")
ax2.legend(fontsize=9)
ax2.set_xticks(layer_ints)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'Layer':<8} {'d_90':<8} {'d_95':<8}")
print("-" * 24)
for layer in layers:
    d = eff_dims[layer]
    marker = " *" if layer in COMMITTED_LAYERS else ""
    print(f"{layer:<8} {d['d90']:<8} {d['d95']:<8}{marker}")

In [0]:
# 6. AUROC comparison: stratified CV vs leave-scenario-out
auroc_cv = [results["stratified_cv"][l]["auroc_mean"] for l in layers]
auroc_cv_std = [results["stratified_cv"][l]["auroc_std"] for l in layers]
auroc_lso = [results["leave_scenario_out_cv"][l]["auroc_mean"] for l in layers]
auroc_lso_std = [results["leave_scenario_out_cv"][l]["auroc_std"] for l in layers]

x = np.arange(len(layers))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, auroc_cv, width, yerr=auroc_cv_std, label="5-fold stratified CV",
       color="#2196F3", alpha=0.8, capsize=4)
ax.bar(x + width/2, auroc_lso, width, yerr=auroc_lso_std, label="Leave-scenario-out CV",
       color="#FF9800", alpha=0.8, capsize=4)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("AUROC")
ax.set_title("Collusion Probe AUROC: Stratified vs Leave-Scenario-Out CV")
ax.set_xticks(x)
ax.set_xticklabels([f"Layer {l}" for l in layers])
ax.legend()
ax.set_ylim(0.3, 1.05)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_auroc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 6b. Layer-wise AUROC profile (bootstrap CIs) + cosine(probe, DIM) by layer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

layer_ints = [int(l) for l in layers]
auroc_means = [bootstrap_cis[l]["mean"] for l in layers]
ci_los = [bootstrap_cis[l]["ci_lo"] for l in layers]
ci_his = [bootstrap_cis[l]["ci_hi"] for l in layers]
yerr_lo = [m - lo for m, lo in zip(auroc_means, ci_los)]
yerr_hi = [hi - m for m, hi in zip(auroc_means, ci_his)]

ax1.errorbar(layer_ints, auroc_means, yerr=[yerr_lo, yerr_hi],
             fmt="o-", capsize=4, color="#2196F3", linewidth=1.5,
             label="5-fold CV (bootstrap 95% CI)")
ax1.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="Chance")
for l in [16, 20, 24]:
    ax1.axvspan(l - 0.3, l + 0.3, alpha=0.08, color="orange")
ax1.set_xlabel("Layer")
ax1.set_ylabel("AUROC")
ax1.set_title("Layer-wise AUROC Profile")
ax1.legend(fontsize=9)
ax1.set_xticks(layer_ints)
ax1.set_ylim(0.4, 0.85)

cosines = [results["directions"][l]["cosine_sim"] for l in layers]
ax2.plot(layer_ints, cosines, "s-", color="#9C27B0", linewidth=1.5)
for l in [16, 20, 24]:
    ax2.axvspan(l - 0.3, l + 0.3, alpha=0.08, color="orange")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Cosine similarity")
ax2.set_title("cos(probe direction, diff-in-means) by Layer")
ax2.set_xticks(layer_ints)
ax2.set_ylim(0.5, 0.8)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_layer_profile.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# 7. Calibration metrics table
from sklearn.metrics import brier_score_loss

print(f"{'Layer':<8} {'Brier':<10} {'ECE':<10} {'Max CE':<10}")
print("-" * 38)
for layer in layers:
    cv = results["stratified_cv"][layer]
    y_true = cv_predictions[layer]["y_true"]
    y_prob = cv_predictions[layer]["y_prob"]

    brier = brier_score_loss(y_true, y_prob)

    bin_edges = np.linspace(0.0, 1.0, 11)
    max_ce = 0.0
    for i in range(10):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob <= hi) if i == 9 else (y_prob >= lo) & (y_prob < hi)
        if mask.sum() > 0:
            gap = abs(y_true[mask].mean() - y_prob[mask].mean())
            max_ce = max(max_ce, gap)

    print(f"{layer:<8} {brier:.4f}    {cv['ece_mean']:.4f}    {max_ce:.4f}")

In [0]:
# 8. Leave-scenario-out per-fold AUROC distribution
fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 4))
if len(layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    lso = results["leave_scenario_out_cv"][layer]
    fold_aurocs = lso["auroc_per_fold"]
    ax.hist(fold_aurocs, bins=10, alpha=0.7, color="#9C27B0", edgecolor="black")
    ax.axvline(lso["auroc_mean"], color="red", linestyle="--",
               label=f"Mean = {lso['auroc_mean']:.3f}")
    ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, label="Chance")
    ax.set_xlabel("AUROC")
    ax.set_ylabel("Count (folds)")
    ax.set_title(f"Layer {layer} — LSO fold distribution")
    ax.legend(fontsize=9)
    ax.set_xlim(-0.05, 1.05)

n_scenarios = results["n_scenarios"]
samples_per_fold = results["n_prompts"] // n_scenarios
plt.suptitle(f"Leave-Scenario-Out AUROC Distribution ({n_scenarios} folds, {samples_per_fold} samples each)", y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "week3_lso_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Observations

Fill this section after the deterministic Colab rerun. Do not assume the rerun will match the earlier standup screenshots.

**Discriminability.** Which layer achieves the highest stratified CV AUROC? Is the signal above chance at the committed layers 16, 20, and 24? Are the bootstrap confidence intervals wide or overlapping?

**Generalization gap.** How does leave-one-scenario-out AUROC compare to stratified CV? Because each scenario contributes only four prompts, treat LSO as a noisy stress test rather than a stable transfer estimate.

**Calibration.** Are predicted probabilities well calibrated? Report Brier score, ECE, max calibration error, and what the reliability diagrams show. If calibration is weak, do not emphasize probability thresholds.

**Geometry.** Does the probe direction align with the difference-in-means direction? Does either direction align with the top principal components, or does the signal sit in a lower-variance subspace?

**Exemplar partitioning.** Does EP recover label-enriched partitions without supervision? If it does not, report that as a geometry result rather than a failure.

**Comparison to prior work.** Goldowsky-Dill et al. report much stronger single-agent deception results on larger models. This development substrate is different: 8B Llama, multi-agent collusion prompts, small sample size, and role labels rather than enacted-behavior labels.


In [0]:
# 9. Generate week3_preliminary_results.md

def _ci(layer):
    c = bootstrap_cis[layer]
    return f"{c['mean']:.3f} [{c['ci_lo']:.3f}, {c['ci_hi']:.3f}]"

auroc_rows = []
for layer in layers:
    ci = bootstrap_cis[layer]
    cv = results["stratified_cv"][layer]
    lso = results["leave_scenario_out_cv"][layer]
    tag = " \\*" if layer in COMMITTED_LAYERS else ""
    auroc_rows.append(
        f"| {layer}{tag} | {_ci(layer)} | {cv['accuracy_mean']:.3f} | "
        f"{cv['brier_mean']:.3f} | {cv['ece_mean']:.3f} | "
        f"{lso['auroc_mean']:.3f} +/- {lso['auroc_std']:.3f} |"
    )

best_5f = max(layers, key=lambda l: bootstrap_cis[l]["mean"])
best_lso = max(layers, key=lambda l: results["leave_scenario_out_cv"][l]["auroc_mean"])
rank_20 = sorted(layers, key=lambda l: -bootstrap_cis[l]["mean"]).index("20") + 1

geom_rows = []
for layer in COMMITTED_LAYERS:
    d = results["directions"][layer]
    ed = eff_dims[layer]
    geom_rows.append(
        f"| {layer} | {d['cosine_sim']:.3f} | {d['probe_norm']:.3f} | "
        f"{d['dim_norm']:.3f} | {ed['d90']} | {ed['d95']} |"
    )

cos_min = min(results["directions"][l]["cosine_sim"] for l in layers)
cos_max = max(results["directions"][l]["cosine_sim"] for l in layers)
ece_min_layer = min(layers, key=lambda l: results["stratified_cv"][l]["ece_mean"])
ece_max_layer = max(layers, key=lambda l: results["stratified_cv"][l]["ece_mean"])
ece_min = results["stratified_cv"][ece_min_layer]["ece_mean"]
ece_max = results["stratified_cv"][ece_max_layer]["ece_mean"]
brier_min = min(results["stratified_cv"][l]["brier_mean"] for l in layers)
brier_max = max(results["stratified_cv"][l]["brier_mean"] for l in layers)
pnorm_13 = results["directions"]["13"]["probe_norm"]
pnorm_24 = results["directions"]["24"]["probe_norm"]
dnorm_13 = results["directions"]["13"]["dim_norm"]
dnorm_24 = results["directions"]["24"]["dim_norm"]
generation_temperature = results.get("generation_temperature", "not recorded")
max_new_tokens = results.get("max_new_tokens", "not recorded")

if ep_results:
    ep_rows = []
    for layer, layer_result in ep_results.get("layers", {}).items():
        partitions = layer_result.get("partitions", [])
        best = min(partitions, key=lambda p: p.get("fisher_p", 1.0)) if partitions else None
        if best:
            ep_rows.append(
                f"| {layer} | {layer_result.get('n_partitions')} | "
                f"{best.get('collusion_rate'):.2f} | {best.get('fisher_p'):.3f} | "
                f"{best.get('cos_probe'):.3f} | {best.get('cos_dim'):.3f} |"
            )
        else:
            ep_rows.append(f"| {layer} | {layer_result.get('n_partitions')} | -- | -- | -- | -- |")
    ep_section = """## Exemplar Partitioning

EP was run as an unsupervised geometry check on selected layers. It asks whether residual-stream directions cluster into partitions that recover colluder versus honest roles without using labels.

| Layer | Partitions | Best colluder rate | Fisher p | cos(ex, probe) | cos(ex, DIM) |
|-------|------------|--------------------|----------|----------------|--------------|
{rows}

Use this table to decide whether EP recovers label-enriched partitions without supervision. If enriched partitions are weak, unstable, or better aligned with DIM than the supervised probe, report EP as geometry evidence rather than as a clean unsupervised detector.
""".format(rows=chr(10).join(ep_rows))
else:
    ep_section = """## Exemplar Partitioning

EP results were not loaded in this run. Week 5 should include EP artifacts from `week2_ep_results.json` before finalizing the handoff summary.
"""

writeup = f"""# Preliminary Results: Linear Probe Detection of Multi-Agent Collusion in Llama 3.1 8B Instruct

## Setup

Model: Llama 3.1 8B Instruct (meta-llama/Llama-3.1-8B-Instruct), loaded via TransformerLens in float16.

Data: {results['n_scenarios']} NARCBench-Core scenarios across 17 domains. Each scenario has 4 committee agents (2 colluder, 2 honest), yielding {results['n_prompts']} prompts with balanced labels. Colluder agents receive hidden instructions to advocate for a specific option; honest agents evaluate on merit. Scenarios use the full NARCBench prompt structure: named agents, role-specific system prompts, option-label randomization, and alternating colluder instruction templates.

Extraction: residual stream activations at the last generated token after a {max_new_tokens}-token response generated at temperature {generation_temperature}, captured via hook_resid_post at layers 13 through 24 (4096-dimensional vectors per prompt).

Probe: logistic regression (C=1.0, L2) with StandardScaler and PCA to {results['pca_components']} components, following Goldowsky-Dill et al. (2025). Primary evaluation: stratified 5-fold CV. Generalization check: leave-one-scenario-out (LSO) CV.

## Discrimination

Table 1 reports AUROC with bootstrap 95% CIs (1000 resamples of pooled 5-fold test predictions).

| Layer | AUROC [95% CI] | Accuracy | Brier | ECE | AUROC (LSO) |
|-------|---------------|----------|-------|-----|-------------|
{chr(10).join(auroc_rows)}

\\* Committed analysis layers (16 ablation, 20 primary, 24 ablation).

The AUROC profile peaks at layer {best_5f} ({_ci(best_5f)}). Layer 20 (pre-registered primary) achieves {_ci("20")}. Layer 24 achieves {_ci("24")}. Use the table above as the source of truth for whether each layer exceeds chance and whether the confidence intervals support a layer-specific claim.

LSO CV should be interpreted cautiously. Each LSO fold tests one scenario (4 data points: 2 colluder, 2 honest), constraining per-fold AUROC to {{0, 0.25, 0.5, 0.75, 1.0}}. Large per-fold variance is a structural artifact of this granularity, so LSO is a noisy stress test rather than a stable cross-scenario transfer estimate.

## Calibration

Calibration should be judged from the reliability diagrams and tables. ECE ranges from {ece_min:.3f} (layer {ece_min_layer}) to {ece_max:.3f} (layer {ece_max_layer}). Brier scores range from {brier_min:.3f} to {brier_max:.3f}; compare these against the 0.250 uninformative baseline for balanced labels.

Use the committed-layer reliability diagrams to report where predicted probabilities deviate from observed frequencies. If calibration is weak, avoid presenting probe scores as trustworthy probabilities.

## Geometry

| Layer | cos(probe, DIM) | Probe norm | DIM norm | d_90 | d_95 |
|-------|----------------|------------|----------|------|------|
{chr(10).join(geom_rows)}

Cosine similarity between the probe direction and the difference-in-means (DIM) direction is stable across layers (range: {cos_min:.3f} to {cos_max:.3f}). The two methods recover overlapping but not identical discriminative axes. Compare these directions against the principal-component alignment table before claiming that the signal occupies a low-variance subspace.

Probe and DIM norms provide a geometry diagnostic, but they should not be interpreted as signal strength by themselves because activation scale can change across layers. Use the AUROC, calibration, and direction-alignment tables together.

For the Phase 0 layer selection: the pre-registered primary layer 20 ranks {rank_20}th of {len(layers)} layers by 5-fold AUROC. Layer {best_5f} achieves the highest discrimination in this run. If the exploratory peak differs from layer 20, raise that as a layer-selection question rather than silently changing the committed layer.

{ep_section}
## Limitations

**Label validity.** Labels reflect role assignment (system prompt), not enacted behavior. A colluder agent may produce honest-sounding text; an honest agent may coincidentally favor the corrupt option. The probe detects the representation of having received collusion instructions, not necessarily the representation of active deception.

**Sample size.** {results['n_scenarios']} scenarios yield {results['n_prompts']} prompts. With PCA to {results['pca_components']} components, the sample-to-feature ratio is {results['n_prompts'] // results['pca_components']}:1. Bootstrap CIs address point-estimate uncertainty but cannot compensate for limited statistical power.

**LSO granularity.** LSO per-fold AUROC is computed on 4 data points per scenario. The resulting metric takes 5 possible values and its standard deviations (0.25 to 0.32) reflect the evaluation structure, not model instability.

**Single model.** Goldowsky-Dill et al. (2025) report AUROC 0.96 to 0.999 for deception probes in 70B+ models. Any performance gap here may reflect the 8B model, the development substrate, the role-label setup, or the difference between multi-agent collusion and single-agent deception.

**Token position.** Activations are extracted at the last generated token after deterministic response generation. Other positions (last input token, mean over generated tokens, or instruction-token positions) may capture different information.

**Rerun consistency.** Use the deterministic `temperature=0.0` rerun as the reference point. Earlier `temperature=0.7` exploratory results should be described only as exploratory and not mixed into final reported numbers.
"""

os.makedirs("results/runway", exist_ok=True)
local_report_path = Path("results/runway/week3_preliminary_results.md")
artifact_report_path = ARTIFACT_DIR / "week3_preliminary_results.md"
with open(local_report_path, "w") as f:
    f.write(writeup)
with open(artifact_report_path, "w") as f:
    f.write(writeup)
print(f"Wrote {local_report_path}")
print(f"Wrote {artifact_report_path}")
print(f"Length: {len(writeup)} chars, {writeup.count(chr(10))} lines")
